# 🧪 PT-W2-D1 概念实验：Entity + Identity 抽取

> 配套阅读：`PT-W2-D1-Entity-Identity抽取.md`
> 从 MI Object Ownership Matrix 的 30+ 对象中，抽取 **本体类型** 和 **业务身份路径**。

## 第 1 格：四种本体类型——dataclass 模拟 Entity 抽取

In [ ]:
from dataclasses import dataclass
from enum import Enum

class ConceptType(Enum):
    ENTITY = "独立实体"           # 有完整生命周期
    RELATIONAL = "关系实体"       # 连接两个实体
    FACT_COMPONENT = "事实构件"   # 不可变事实记录
    ACTIVITY = "活动实体"         # 有开始和结束

@dataclass(frozen=True)
class OntologyConcept:
    name: str
    concept_type: ConceptType
    owner_context: str
    tech_id_field: str

# P0 核心对象（摘自 §3.1）
concepts = [
    OntologyConcept("Resource Unit", ConceptType.ENTITY, "Asset Foundation", "resource_code"),
    OntologyConcept("Merchant", ConceptType.ENTITY, "Merchant", "legal_identity"),
    OntologyConcept("Contract", ConceptType.ENTITY, "Contract Lifecycle", "contract_id"),
    OntologyConcept("Occupancy", ConceptType.RELATIONAL, "Lease/Occupancy", "occupancy_id"),
    OntologyConcept("Revenue Evidence", ConceptType.FACT_COMPONENT, "Operations", "evidence_id"),
    OntologyConcept("Energy Reading", ConceptType.FACT_COMPONENT, "Engineering", "reading_id"),
    OntologyConcept("Operation Task", ConceptType.ACTIVITY, "Operations", "task_id"),
    OntologyConcept("Service Ticket", ConceptType.ACTIVITY, "Work Order", "ticket_id"),
    OntologyConcept("Bill/AR", ConceptType.ENTITY, "Billing & AR", "bill_id"),
    OntologyConcept("Contract Clause", ConceptType.FACT_COMPONENT, "Contract", "clause_id"),
]

from collections import Counter
counts = Counter(c.concept_type for c in concepts)
for t, n in counts.items():
    print(f"  {t.value}: {n} 个")

## 第 2 格：业务身份路径——D-014 唯一真值来源

In [ ]:
@dataclass
class BusinessIdentity:
    concept_name: str
    instance_id: str
    identity_path: tuple[str, ...]  # 业务身份路径
    truth_owner: str                # D-014 唯一真值来源

# 身份路径：空间层级 / 证件 / 时间
identities = [
    BusinessIdentity("Resource Unit", "A101",
                     ("龙湖天街", "A栋", "1F", "A101"), "Asset Foundation"),
    BusinessIdentity("Merchant", "M-001",
                     ("统一社会信用代码", "91310000..."), "Merchant"),
    BusinessIdentity("Contract", "CT2026001",
                     ("M-001", "A101", "2026-08-01"), "Contract Lifecycle"),
    BusinessIdentity("Occupancy", "OCC-001",
                     ("A101", "2026-01~2026-12"), "Lease/Occupancy"),
]

for ident in identities:
    path = " → ".join(ident.identity_path)
    print(f"{ident.concept_name} [{ident.instance_id}]")
    print(f"  路径: {path}")
    print(f"  真值主人: {ident.truth_owner}  (D-014)")
    print()

## 第 3 格：可视化——本体类型分布

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc")
font_name = font_manager.FontProperties(fname="/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc").get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

labels = [t.value for t in ConceptType]
sizes = [counts[t] for t in ConceptType]
colors = ["#4CAF50", "#2196F3", "#FF9800", "#9C27B0"]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, sizes, color=colors, edgecolor="white")
ax.set_title("MI CRE Ontology 概念类型分布 (P0 核心)")
ax.set_ylabel("数量")
for bar, s in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(s), ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig("/root/learning-notebooks/第10周/d1_entity_types.png", dpi=100)
plt.show()
print("本体类型分布已绘制")

## 第 4 格：D-014 验证——跨 Context 真值唯一性

Agent 问"铺位 A101 的营业额"应去 Operations 查，不去 Billing 查。

In [ ]:
# D-014：每个业务事实只有一个 Owner Context
truth_ownership = {
    ("Resource Unit", "物理状态"): "Asset Foundation",
    ("Resource Unit", "占用状态"): "Lease/Occupancy",
    ("Resource Unit", "计费状态"): "Billing & AR",
    ("营业额", "采集证据"): "Operations",
    ("合同条款", "真值"): "Contract",
}

def query_truth(concept, fact):
    owner = truth_ownership.get((concept, fact))
    if owner:
        return f"查询 [{concept}/{fact}] → 去 Owner: {owner} 查"
    return f"未知事实 [{concept}/{fact}]"

print(query_truth("Resource Unit", "物理状态"))
print(query_truth("Resource Unit", "占用状态"))
print(query_truth("Resource Unit", "计费状态"))
print(query_truth("营业额", "采集证据"))
print()
print("结论：Agent 跨 Context 推理时不会遇到矛盾数据（D-014 身份唯一性规则）")